# Feature Selection, ROC/PR Curves & SHAP Explainability
Evaluates collinearity pruning, out-of-fold ROC and Precision-Recall performance, and global SHAP attributions on 3D glioma radiomic features.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.impute import SimpleImputer
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay, roc_auc_score, average_precision_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

df = pd.read_csv(Path("../data/processed/radiomic_features.csv"))
feature_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != "label"]
corr_matrix = df[feature_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.90)]
selected_cols = [c for c in feature_cols if c not in to_drop]
print(f"Total features: {len(feature_cols)} | Dropped (>0.90 corr): {len(to_drop)} | Selected: {len(selected_cols)}")
X = df[selected_cols].copy()
y = df["label"].to_numpy()

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros(len(y))
for train_idx, val_idx in skf.split(X, y):
    pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("clf", XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42, n_jobs=-1))])
    pipe.fit(X.iloc[train_idx], y[train_idx])
    oof_probs[val_idx] = pipe.predict_proba(X.iloc[val_idx])[:, 1]

print(f"OOF ROC-AUC: {roc_auc_score(y, oof_probs):.3f} | OOF PR-AUC: {average_precision_score(y, oof_probs):.3f}")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(y, oof_probs, ax=ax1, name="XGBoost (5-fold OOF)")
ax1.set_title("ROC Curve")
PrecisionRecallDisplay.from_predictions(y, oof_probs, ax=ax2, name="XGBoost (5-fold OOF)")
ax2.set_title("Precision-Recall Curve")
plt.tight_layout()
plt.show()

In [ ]:
final_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler()), ("clf", XGBClassifier(n_estimators=150, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42, n_jobs=-1))])
final_pipe.fit(X, y)
X_proc = pd.DataFrame(final_pipe[:-1].transform(X), columns=X.columns, index=X.index)
explainer = shap.TreeExplainer(final_pipe.named_steps["clf"])
shap_values = explainer.shap_values(X_proc)
shap.summary_plot(shap_values, X_proc, max_display=12)